# Lab 2 — Premium MCP Inventory Management System with Gradio

**Course:** PGD Data Science with AI & Generative AI  
**Topic:** Model Context Protocol (MCP)  
**Frontend:** Premium Gradio Operations Console  
**Backend:** SQLite + MCP Python SDK

## Business Scenario
You are building an AI-enabled **Retail Operations Console** for inventory and order management.

**Architecture:** `User → Gradio UI → MCP Client → MCP Server → SQLite`

The UI is the experience layer. MCP is the controlled capability layer.

### Students will build
- product search,
- SKU inventory inspection,
- low-stock monitoring,
- verified order tracking,
- controlled restock requests,
- a support-assistant workflow,
- and an MCP capability explorer.

## Learning Outcomes

By the end of this lab, students should be able to:

1. Build MCP Tools, Resources, and Prompts.
2. Connect an MCP Client to the server.
3. Return structured tool output with Pydantic.
4. Build a premium Gradio `Blocks` interface.
5. Call MCP capabilities from Gradio events.
6. Separate read-only tools from state-changing tools.
7. Apply human approval to high-impact actions.

In [ ]:
# Install once in Google Colab / Jupyter
%pip -q install "mcp>=2,<3" "gradio>=6,<7" pandas pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import sqlite3
import json
from datetime import datetime, timezone

import pandas as pd
import gradio as gr
from pydantic import BaseModel

from mcp import Client
from mcp.server import MCPServer
from mcp.types import TextContent, TextResourceContents

print("Gradio version:", gr.__version__)
print("Environment ready.")

Gradio version: 6.20.0
Environment ready.


# 1. Create the RetailOps Operational Database

For teaching, SQLite acts as the operational system. A production version could connect to PostgreSQL, SAP/ERP, Shopify, a WMS, or internal APIs.

In [ ]:
DB_PATH = Path("retail_ops_premium.db")

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

with get_connection() as conn:
    conn.executescript('''
    DROP TABLE IF EXISTS products;
    DROP TABLE IF EXISTS orders;
    DROP TABLE IF EXISTS restock_requests;

    CREATE TABLE products (
        sku TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        stock INTEGER NOT NULL,
        reorder_level INTEGER NOT NULL
    );

    CREATE TABLE orders (
        order_id TEXT PRIMARY KEY,
        customer_name TEXT NOT NULL,
        sku TEXT NOT NULL,
        quantity INTEGER NOT NULL,
        status TEXT NOT NULL,
        courier TEXT,
        tracking_no TEXT,
        expected_delivery TEXT,
        FOREIGN KEY (sku) REFERENCES products(sku)
    );

    CREATE TABLE restock_requests (
        request_id INTEGER PRIMARY KEY AUTOINCREMENT,
        sku TEXT NOT NULL,
        requested_qty INTEGER NOT NULL,
        reason TEXT NOT NULL,
        created_at TEXT NOT NULL,
        approved INTEGER NOT NULL DEFAULT 0,
        FOREIGN KEY (sku) REFERENCES products(sku)
    );
    ''')

    conn.executemany(
        "INSERT INTO products VALUES (?, ?, ?, ?, ?, ?)",
        [
            ("LAP-101", "NovaBook Pro 14", "Laptop", 1299.00, 7, 10),
            ("LAP-120", "NovaBook Air 13", "Laptop", 999.00, 18, 8),
            ("MON-220", "VisionView 27 Monitor", "Monitor", 329.00, 22, 8),
            ("MON-240", "VisionView UltraWide 34", "Monitor", 549.00, 6, 7),
            ("KEY-310", "TypeFast Mechanical Keyboard", "Accessory", 89.00, 4, 12),
            ("KEY-320", "TypeFast Mini Keyboard", "Accessory", 69.00, 26, 10),
            ("MOU-410", "Precision Wireless Mouse", "Accessory", 49.00, 35, 10),
            ("HDP-510", "SoundMax Pro Headphones", "Audio", 149.00, 9, 10),
            ("HDP-520", "SoundMax Studio Headset", "Audio", 199.00, 17, 8),
            ("CAM-610", "VisionCam 4K", "Camera", 139.00, 5, 9),
            ("DOC-710", "USB-C Dock Pro", "Accessory", 179.00, 14, 8),
            ("SSD-810", "RapidStore 2TB SSD", "Storage", 189.00, 11, 12),
        ],
    )

    conn.executemany(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        [
            ("ORD-1001", "Aisha Khan", "MON-220", 1, "Delivered", "DHL", "DHL778801", "2026-08-10"),
            ("ORD-1002", "Bilal Ahmed", "LAP-101", 1, "Shipped", "FedEx", "FDX982145", "2026-08-17"),
            ("ORD-1003", "Sara Ali", "KEY-310", 2, "Processing", None, None, "2026-08-18"),
            ("ORD-1004", "Omar Hassan", "HDP-510", 1, "Packed", None, None, "2026-08-16"),
            ("ORD-1005", "Mariam Noor", "CAM-610", 1, "Shipped", "DHL", "DHL811204", "2026-08-19"),
            ("ORD-1006", "Hamza Tariq", "DOC-710", 2, "Processing", None, None, "2026-08-20"),
        ],
    )

print("Database created:", DB_PATH.resolve())

Database created: /content/retail_ops_premium.db


In [ ]:
with get_connection() as conn:
    display(pd.read_sql_query("SELECT * FROM products", conn))
    display(pd.read_sql_query("SELECT * FROM orders", conn))

,sku,name,category,price,stock,reorder_level
0,LAP-101,NovaBook Pro 14,Laptop,1299.0,7,10
1,LAP-120,NovaBook Air 13,Laptop,999.0,18,8
2,MON-220,VisionView 27 Monitor,Monitor,329.0,22,8
3,MON-240,VisionView UltraWide 34,Monitor,549.0,6,7
4,KEY-310,TypeFast Mechanical Keyboard,Accessory,89.0,4,12
5,KEY-320,TypeFast Mini Keyboard,Accessory,69.0,26,10
6,MOU-410,Precision Wireless Mouse,Accessory,49.0,35,10
7,HDP-510,SoundMax Pro Headphones,Audio,149.0,9,10
8,HDP-520,SoundMax Studio Headset,Audio,199.0,17,8
9,CAM-610,VisionCam 4K,Camera,139.0,5,9


,order_id,customer_name,sku,quantity,status,courier,tracking_no,expected_delivery
0,ORD-1001,Aisha Khan,MON-220,1,Delivered,DHL,DHL778801,2026-08-10
1,ORD-1002,Bilal Ahmed,LAP-101,1,Shipped,FedEx,FDX982145,2026-08-17
2,ORD-1003,Sara Ali,KEY-310,2,Processing,None,None,2026-08-18
3,ORD-1004,Omar Hassan,HDP-510,1,Packed,None,None,2026-08-16
4,ORD-1005,Mariam Noor,CAM-610,1,Shipped,DHL,DHL811204,2026-08-19
5,ORD-1006,Hamza Tariq,DOC-710,2,Processing,None,None,2026-08-20


# 2. Structured Outputs and MCP Server

In [ ]:
class InventoryStatus(BaseModel):
    sku: str
    product: str
    category: str
    price: float
    stock: int
    reorder_level: int
    stock_gap: int
    status: str

class OrderInfo(BaseModel):
    order_id: str
    customer_name: str
    sku: str
    product: str
    quantity: int
    status: str
    courier: str | None = None
    tracking_no: str | None = None
    expected_delivery: str

class RestockRequestResult(BaseModel):
    request_id: int
    sku: str
    product: str
    requested_qty: int
    reason: str
    approved: bool
    message: str

mcp = MCPServer(
    "RetailOps Premium MCP",
    instructions=(
        "Verify inventory and order facts using RetailOps tools. "
        "Restock requests create pending records and require human approval."
    ),
)
print("MCP server created.")

MCP server created.


# 3. Register MCP Tools

In [ ]:
@mcp.tool(title="Search product catalog")
def search_products(query: str = "", limit: int = 20) -> list[dict]:
    """Search products by SKU, name, or category."""
    if limit < 1 or limit > 100:
        raise ValueError("limit must be between 1 and 100.")
    term = f"%{query.strip()}%"
    with get_connection() as conn:
        rows = conn.execute(
            '''SELECT sku, name, category, price, stock, reorder_level
               FROM products
               WHERE sku LIKE ? OR name LIKE ? OR category LIKE ?
               ORDER BY name LIMIT ?''',
            (term, term, term, limit),
        ).fetchall()
    out = []
    for row in rows:
        item = dict(row)
        item["status"] = "REORDER REQUIRED" if item["stock"] <= item["reorder_level"] else "HEALTHY"
        out.append(item)
    return out

@mcp.tool(title="Check inventory")
def check_inventory(sku: str) -> InventoryStatus:
    """Return stock health and reorder status for one SKU."""
    sku = sku.strip().upper()
    with get_connection() as conn:
        row = conn.execute(
            "SELECT sku, name, category, price, stock, reorder_level FROM products WHERE sku = ?",
            (sku,),
        ).fetchone()
    if row is None:
        raise ValueError(f"Unknown SKU: {sku}")
    stock, level = int(row["stock"]), int(row["reorder_level"])
    status = "OUT OF STOCK" if stock == 0 else "REORDER REQUIRED" if stock <= level else "WATCH" if stock <= level * 1.5 else "HEALTHY"
    return InventoryStatus(
        sku=row["sku"], product=row["name"], category=row["category"],
        price=float(row["price"]), stock=stock, reorder_level=level,
        stock_gap=max(0, level-stock), status=status,
    )

@mcp.tool(title="Get order status")
def get_order_status(order_id: str) -> OrderInfo:
    """Return verified order and fulfillment information."""
    order_id = order_id.strip().upper()
    with get_connection() as conn:
        row = conn.execute(
            '''SELECT o.order_id, o.customer_name, o.sku, p.name AS product,
                      o.quantity, o.status, o.courier, o.tracking_no, o.expected_delivery
               FROM orders o JOIN products p ON p.sku = o.sku
               WHERE o.order_id = ?''',
            (order_id,),
        ).fetchone()
    if row is None:
        raise ValueError(f"Order {order_id} was not found.")
    return OrderInfo(**dict(row))

@mcp.tool(title="Low stock report")
def low_stock_report() -> list[dict]:
    """Return all products at or below reorder level."""
    with get_connection() as conn:
        rows = conn.execute(
            '''SELECT sku, name, category, stock, reorder_level
               FROM products WHERE stock <= reorder_level
               ORDER BY (CAST(stock AS REAL)/NULLIF(reorder_level,0)), stock'''
        ).fetchall()
    report=[]
    for row in rows:
        level, stock = int(row["reorder_level"]), int(row["stock"])
        report.append({
            "sku":row["sku"], "product":row["name"], "category":row["category"],
            "stock":stock, "reorder_level":level,
            "suggested_reorder_qty":max(0, level*3-stock),
            "priority":"CRITICAL" if stock <= max(1, level//2) else "HIGH",
        })
    return report

@mcp.tool(title="Create restock request")
def create_restock_request(sku: str, requested_qty: int, reason: str="Low inventory") -> RestockRequestResult:
    """Create an unapproved restock request for human review."""
    sku, reason = sku.strip().upper(), reason.strip()
    if requested_qty < 1 or requested_qty > 5000:
        raise ValueError("requested_qty must be between 1 and 5000.")
    with get_connection() as conn:
        product = conn.execute("SELECT sku, name FROM products WHERE sku=?", (sku,)).fetchone()
        if product is None:
            raise ValueError(f"Unknown SKU: {sku}")
        cur = conn.execute(
            '''INSERT INTO restock_requests(sku,requested_qty,reason,created_at,approved)
               VALUES(?,?,?,?,0)''',
            (sku, int(requested_qty), reason, datetime.now(timezone.utc).isoformat()),
        )
        conn.commit()
        request_id = int(cur.lastrowid)
    return RestockRequestResult(
        request_id=request_id, sku=sku, product=product["name"],
        requested_qty=int(requested_qty), reason=reason, approved=False,
        message="Pending human approval.",
    )

# 4. Register MCP Resources and Prompts

In [ ]:
@mcp.resource("retail://policy/returns")
def return_policy() -> str:
    return """RETAILOPS RETURN POLICY
• Standard products may be returned within 14 days of delivery.
• Products should be in original condition with packaging.
• Opened headphones are non-returnable for hygiene reasons unless defective.
• Damaged or incorrect items should be reported within 48 hours.
• Refunds are processed after inspection.
• High-value laptop returns require manual support review.
""".strip()

@mcp.resource("retail://policy/restocking")
def restocking_policy() -> str:
    return """RETAILOPS RESTOCKING POLICY
• Products at or below reorder level should be reviewed.
• MCP can create a pending request but cannot approve procurement.
• Requested quantity should be justified by current and target stock.
• High-value or unusually large requests require management approval.
• All restock actions must remain auditable.
""".strip()

@mcp.resource("retail://product/{sku}")
def product_resource(sku: str) -> str:
    sku=sku.strip().upper()
    with get_connection() as conn:
        row=conn.execute("SELECT * FROM products WHERE sku=?",(sku,)).fetchone()
    if row is None:
        raise ValueError(f"Unknown SKU: {sku}")
    return json.dumps(dict(row), indent=2)

@mcp.prompt(title="Customer support response")
def customer_support_response(issue: str, order_id: str) -> str:
    return f"""You are a professional RetailOps support assistant.
Customer issue: {issue}
Order ID: {order_id}
Verify order facts through MCP. Read the returns policy when needed.
Never invent courier names, tracking numbers, delivery dates, or policies.
Keep the response concise and clearly based on verified evidence.
""".strip()

@mcp.prompt(title="Inventory manager brief")
def inventory_manager_brief(priority: str="High") -> str:
    return f"""Prepare an inventory-management brief with priority {priority}.
Use verified MCP results only. Include low-stock items, reorder levels,
suggested quantities, pending requests, risks, and actions needing approval.
""".strip()

# 5. Test MCP Discovery Before Building the UI

In [ ]:
async with Client(mcp) as client:
    print("TOOLS:", [t.name for t in (await client.list_tools()).tools])
    print("RESOURCES:", [str(r.uri) for r in (await client.list_resources()).resources])
    print("PROMPTS:", [p.name for p in (await client.list_prompts()).prompts])

TOOLS: ['search_products', 'check_inventory', 'get_order_status', 'low_stock_report', 'create_restock_request']
RESOURCES: ['retail://policy/returns', 'retail://policy/restocking']
PROMPTS: ['customer_support_response', 'inventory_manager_brief']


# 6. Gradio Callback Helpers

Every important UI action below goes through the MCP Client.

In [ ]:
def unwrap_mcp_structured(value):
    """
    Normalize MCP structured_content for Gradio/Pandas.

    MCP SDK may wrap a list return value as:
        {"result": [ {...}, {...} ]}

    A direct Pydantic/dict result may instead look like:
        {"sku": "...", "stock": 4, ...}

    This helper safely unwraps common wrapper keys without destroying
    normal business dictionaries.
    """
    if value is None:
        return None

    # Common MCP/list wrappers.
    if isinstance(value, dict):
        for key in ("result", "results", "items", "data"):
            if (
                key in value
                and len(value) == 1
                and isinstance(value[key], (list, tuple))
            ):
                return value[key]

        # Some SDK/tool combinations may return {"result": {...}}
        if (
            "result" in value
            and len(value) == 1
            and isinstance(value["result"], dict)
        ):
            return value["result"]

    return value


def to_df(value):
    value = unwrap_mcp_structured(value)

    if isinstance(value, pd.DataFrame):
        return value.copy()

    if isinstance(value, (list, tuple)):
        if not value:
            return pd.DataFrame()

        # Normal list of records.
        if all(isinstance(x, dict) for x in value):
            return pd.DataFrame(list(value))

        # Scalar list fallback.
        return pd.DataFrame({"value": list(value)})

    if isinstance(value, dict):
        return pd.DataFrame([value])

    if value is None:
        return pd.DataFrame()

    return pd.DataFrame([{"value": value}])
def status_badge(status):
    status=str(status).upper()
    cls="ok" if "HEALTHY" in status or "DELIVERED" in status else "warn" if "WATCH" in status or "PROCESS" in status or "PACKED" in status else "danger" if "REORDER" in status or "CRITICAL" in status or "OUT" in status else "neutral"
    return f"<span class='pill {cls}'>{status}</span>"

def dashboard_html():
    with get_connection() as conn:
        p=conn.execute("SELECT COUNT(*) c FROM products").fetchone()["c"]
        low=conn.execute("SELECT COUNT(*) c FROM products WHERE stock<=reorder_level").fetchone()["c"]
        units=conn.execute("SELECT SUM(stock) c FROM products").fetchone()["c"]
        pending=conn.execute("SELECT COUNT(*) c FROM restock_requests WHERE approved=0").fetchone()["c"]
        value=conn.execute("SELECT SUM(price*stock) c FROM products").fetchone()["c"]
        active=conn.execute("SELECT COUNT(*) c FROM orders WHERE status NOT IN ('Delivered','Cancelled')").fetchone()["c"]
    vals=[("Catalog",p,"Active SKUs","▦"),("Low Stock",low,"Need attention","!"),("Units",units,"On hand","▤"),("Restock",pending,"Pending approval","↻"),("Stock Value",f"${value:,.0f}","Estimated","$"),("Orders",active,"In fulfillment","→")]
    return "<div class='kpis'>"+"".join(f"<div class='kpi'><div class='kicon'>{i}</div><div><small>{a}</small><b>{b}</b><span>{c}</span></div></div>" for a,b,c,i in vals)+"</div>"

def restock_table():
    with get_connection() as conn:
        return pd.read_sql_query('''SELECT r.request_id,r.sku,p.name AS product,r.requested_qty,r.reason,r.created_at,
            CASE WHEN r.approved=1 THEN 'APPROVED' ELSE 'PENDING' END AS status
            FROM restock_requests r JOIN products p ON p.sku=r.sku ORDER BY r.request_id DESC''', conn)

async def ui_search(query):
    async with Client(mcp) as c:
        r=await c.call_tool("search_products",{"query":query or "","limit":50})

    if r.is_error:
        msg = r.content[0].text if r.content else "Product search failed."
        return pd.DataFrame(), f"⚠️ {msg}"

    df = to_df(r.structured_content)

    preferred = [
        "sku", "name", "category", "price",
        "stock", "reorder_level", "status"
    ]
    if not df.empty:
        cols = [c for c in preferred if c in df.columns]
        if cols:
            df = df[cols]

    return df, f"**{len(df)}** product(s) found."

async def ui_inventory(sku):
    async with Client(mcp) as c:
        r=await c.call_tool("check_inventory",{"sku":(sku or "").upper()})
    if r.is_error:
        return "<div class='error'>Invalid or unknown SKU.</div>", pd.DataFrame()
    d=r.structured_content
    html=f"""<div class='detail'><div class='detailtop'><div><small>SKU {d['sku']}</small><h3>{d['product']}</h3><span>{d['category']}</span></div>{status_badge(d['status'])}</div>
    <div class='metrics'><div><small>Price</small><b>${d['price']:,.2f}</b></div><div><small>Stock</small><b>{d['stock']}</b></div><div><small>Reorder level</small><b>{d['reorder_level']}</b></div><div><small>Gap</small><b>{d['stock_gap']}</b></div></div></div>"""
    return html, to_df(d)

async def ui_low():
    async with Client(mcp) as c:
        r=await c.call_tool("low_stock_report",{})

    if r.is_error:
        msg = r.content[0].text if r.content else "Low-stock report failed."
        return pd.DataFrame(), f"### Replenishment attention\n⚠️ {msg}"

    df = to_df(r.structured_content)

    preferred = [
        "sku", "product", "category", "stock",
        "reorder_level", "suggested_reorder_qty", "priority"
    ]
    if not df.empty:
        cols = [c for c in preferred if c in df.columns]
        if cols:
            df = df[cols]

    return (
        df,
        f"### Replenishment attention\n"
        f"**{len(df)}** SKU(s) are at or below reorder level."
    )

async def ui_order(order_id):
    async with Client(mcp) as c:
        r=await c.call_tool("get_order_status",{"order_id":(order_id or "").upper()})
    if r.is_error:
        return "<div class='error'>Order not found.</div>", pd.DataFrame()
    d=r.structured_content
    html=f"""<div class='detail'><div class='detailtop'><div><small>{d['order_id']}</small><h3>{d['customer_name']}</h3><span>{d['product']}</span></div>{status_badge(d['status'])}</div>
    <div class='metrics six'><div><small>SKU</small><b>{d['sku']}</b></div><div><small>Qty</small><b>{d['quantity']}</b></div><div><small>Courier</small><b>{d.get('courier') or 'Not assigned'}</b></div><div><small>Tracking</small><b>{d.get('tracking_no') or 'N/A'}</b></div><div><small>Expected</small><b>{d['expected_delivery']}</b></div></div></div>"""
    return html, to_df(d)

async def ui_restock(sku,qty,reason):
    try: qty=int(qty)
    except: return "<div class='error'>Quantity must be a whole number.</div>",restock_table(),dashboard_html()
    async with Client(mcp) as c:
        r=await c.call_tool("create_restock_request",{"sku":(sku or "").upper(),"requested_qty":qty,"reason":reason})
    if r.is_error: return "<div class='error'>Request could not be created.</div>",restock_table(),dashboard_html()
    d=r.structured_content
    return f"<div class='success'><b>✓ Request #{d['request_id']} created</b><br>{d['product']} · {d['requested_qty']} units<br><small>Pending human approval</small></div>",restock_table(),dashboard_html()

async def ui_support(order_id, issue):
    async with Client(mcp) as c:
        order=await c.call_tool("get_order_status",{"order_id":order_id.upper()})
        policy=await c.read_resource("retail://policy/returns")
        prompt=await c.get_prompt("customer_support_response",{"issue":issue,"order_id":order_id})
    if order.is_error: return "Order could not be verified.",""
    d=order.structured_content
    ptxt="\n".join(x.text for x in policy.contents if isinstance(x,TextResourceContents))
    prtxt="\n".join(x.content.text for x in prompt.messages if isinstance(x.content,TextContent))
    answer=f"""Hello {d['customer_name']},\n\nYour order **{d['order_id']}** for **{d['product']}** is currently **{d['status']}**. Expected delivery: **{d['expected_delivery']}**. {('Tracking: **'+d['tracking_no']+'**.') if d.get('tracking_no') else ''}\n\nEligible standard products may be returned within **14 days of delivery** in original condition with packaging. High-value laptop returns require manual review.\n\n_This answer uses verified MCP data._"""
    evidence=f"""### MCP Evidence\n```json\n{json.dumps(d,indent=2)}\n```\n\n**Policy**\n```text\n{ptxt}\n```\n\n**Rendered prompt**\n```text\n{prtxt}\n```"""
    return answer,evidence

async def ui_explorer():
    async with Client(mcp) as c:
        tools=await c.list_tools(); resources=await c.list_resources(); templates=await c.list_resource_templates(); prompts=await c.list_prompts()
    rows=[]
    rows += [{"type":"Tool","name":x.name,"description":x.description or ""} for x in tools.tools]
    rows += [{"type":"Resource","name":str(x.uri),"description":getattr(x,"description","") or ""} for x in resources.resources]
    rows += [{"type":"Resource Template","name":str(x.uri_template),"description":getattr(x,"description","") or ""} for x in templates.resource_templates]
    rows += [{"type":"Prompt","name":x.name,"description":x.description or ""} for x in prompts.prompts]
    arch="<div class='arch'><div>Gradio UI</div><b>→</b><div>MCP Client</div><b>→</b><div class='active'>MCP Server</div><b>→</b><div>SQLite</div></div>"
    return pd.DataFrame(rows),arch

In [ ]:
# Quick normalization test.
# This reproduces the wrapper shape that previously caused [object Object].
_mock_wrapped_result = {
    "result": [
        {
            "sku": "KEY-310",
            "product": "TypeFast Mechanical Keyboard",
            "stock": 4,
            "reorder_level": 12,
            "suggested_reorder_qty": 32,
            "priority": "CRITICAL",
        },
        {
            "sku": "LAP-101",
            "product": "NovaBook Pro 14",
            "stock": 7,
            "reorder_level": 10,
            "suggested_reorder_qty": 23,
            "priority": "HIGH",
        },
    ]
}

_test_df = to_df(_mock_wrapped_result)
display(_test_df)

assert len(_test_df) == 2
assert "sku" in _test_df.columns
assert "result" not in _test_df.columns

print("MCP → Pandas normalization test passed.")

,sku,product,stock,reorder_level,suggested_reorder_qty,priority
0,KEY-310,TypeFast Mechanical Keyboard,4,12,32,CRITICAL
1,LAP-101,NovaBook Pro 14,7,10,23,HIGH


MCP → Pandas normalization test passed.


# 7. Premium Gradio UI

The styling below creates a high-end operations console while keeping the implementation understandable for students.

> **Fix included:** MCP list results are unwrapped before being sent to `gr.Dataframe`, preventing `[object Object]` rendering.

In [ ]:
CSS=r'''
.gradio-container{max-width:1500px!important;margin:auto!important;background:radial-gradient(circle at 85% 4%,rgba(22,104,227,.12),transparent 26%),linear-gradient(#fbfdff,#f2f6fb)!important}
.hero{padding:34px 38px;border-radius:24px;background:radial-gradient(circle at 84% 20%,rgba(60,170,255,.32),transparent 22%),linear-gradient(135deg,#061932,#0b315e 62%,#125388);color:white;margin-bottom:16px;overflow:hidden}.hero h1{font-size:40px;color:white!important;margin:4px 0}.hero p{max-width:780px;color:#c9d9ea}.badge{display:inline-block;padding:7px 12px;border:1px solid #ffffff30;border-radius:999px;background:#ffffff10;font-size:12px;letter-spacing:.08em}.chips{display:flex;gap:8px;flex-wrap:wrap;margin-top:18px}.chips span{font-size:11px;padding:6px 9px;border-radius:9px;background:#ffffff12}
.kpis{display:grid;grid-template-columns:repeat(6,1fr);gap:12px;margin-bottom:18px}.kpi{display:flex;gap:12px;align-items:center;padding:16px;background:white;border:1px solid #dce6f0;border-radius:18px;box-shadow:0 8px 24px #12395b0d}.kicon{width:40px;height:40px;display:grid;place-items:center;border-radius:12px;background:#edf4ff;color:#1668e3;font-weight:800}.kpi small,.kpi span{display:block;color:#708399;font-size:11px}.kpi b{display:block;color:#071a34;font-size:23px;margin:3px 0}
.detail{background:white;border:1px solid #dce6f0;border-radius:20px;padding:20px;min-height:220px}.detailtop{display:flex;justify-content:space-between;border-bottom:1px solid #edf1f6;padding-bottom:15px}.detailtop small{color:#1668e3;font-weight:700;letter-spacing:.08em}.detailtop h3{margin:4px 0;color:#071a34;font-size:25px}.detailtop span{color:#708399}.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-top:15px}.metrics.six{grid-template-columns:repeat(5,1fr)}.metrics div{padding:12px;border-radius:12px;background:#f7f9fc;border:1px solid #edf1f6}.metrics small{display:block;color:#718499}.metrics b{color:#071a34}.pill{padding:7px 10px;border-radius:999px;font-size:10px;font-weight:800}.pill.ok{background:#e8f8f1;color:#087654}.pill.warn{background:#fff4dc;color:#9a6300}.pill.danger{background:#ffedef;color:#b83243}.pill.neutral{background:#eef3f8;color:#516579}
.error,.success{padding:16px;border-radius:14px}.error{background:#fff2f3;border:1px solid #f0c5ca;color:#aa3040}.success{background:#eefaf5;border:1px solid #bfe6d7;color:#12694e}.arch{display:flex;gap:9px;align-items:center;flex-wrap:wrap}.arch div{padding:14px;border:1px solid #d8e3ef;background:white;border-radius:13px;min-width:140px;text-align:center}.arch .active{background:#eef6ff;border-color:#9fc6fa}.arch b{color:#1668e3;font-size:22px}
.gradio-container button.primary{background:linear-gradient(135deg,#155fc9,#1779e6)!important;border:none!important}.gradio-container input,.gradio-container textarea{border-radius:12px!important}footer{display:none!important}
@media(max-width:1100px){.kpis{grid-template-columns:repeat(3,1fr)}}@media(max-width:700px){.kpis{grid-template-columns:repeat(2,1fr)}.metrics,.metrics.six{grid-template-columns:repeat(2,1fr)}.hero h1{font-size:30px}}
'''

HERO="""<div class='hero'><div class='badge'>● MCP CONNECTED · RETAIL OPERATIONS</div><h1>RetailOps Intelligence Console</h1><p>Premium inventory and order operations workspace powered by Model Context Protocol. Verify data, monitor inventory risk, create controlled actions, and inspect the MCP capability layer.</p><div class='chips'><span>MCP Server: RetailOps Premium MCP</span><span>SQLite backend</span><span>Gradio frontend</span><span>Human approval for restocking</span></div></div>"""

theme=gr.themes.Soft(primary_hue="blue",secondary_hue="slate",neutral_hue="slate")

with gr.Blocks(theme=theme,css=CSS,title="RetailOps MCP Intelligence Console") as demo:
    gr.HTML(HERO)
    dashboard=gr.HTML(dashboard_html())

    with gr.Tabs():
        with gr.Tab("Overview"):
            gr.Markdown("## Inventory Health at a Glance\nLive operational KPIs plus MCP-generated low-stock intelligence.")
            refresh=gr.Button("↻ Refresh Dashboard")
            with gr.Row():
                low_summary=gr.Markdown()
                low_overview=gr.Dataframe(interactive=False,wrap=True,scale=3)

        with gr.Tab("Product Search"):
            gr.Markdown("## Product Catalog Search\n**MCP Tool:** `search_products(query, limit)`")
            with gr.Row():
                q=gr.Textbox(label="Search",placeholder="laptop, Accessory, LAP-101 ...",scale=4)
                qbtn=gr.Button("Search Products",variant="primary",scale=1)
            qmsg=gr.Markdown(); qdf=gr.Dataframe(interactive=False,wrap=True)

        with gr.Tab("Inventory Inspector"):
            gr.Markdown("## SKU Inventory Inspector\n**MCP Tool:** `check_inventory(sku)`")
            with gr.Row():
                with gr.Column(scale=1):
                    sku=gr.Textbox(label="SKU",placeholder="KEY-310")
                    skubtn=gr.Button("Inspect Inventory",variant="primary")
                    gr.Examples([["KEY-310"],["MON-220"],["CAM-610"]],sku)
                with gr.Column(scale=2):
                    invcard=gr.HTML("<div class='detail'>Enter a SKU to inspect stock health.</div>")
            invraw=gr.Dataframe(label="Structured MCP Result",interactive=False)

        with gr.Tab("Low-Stock Intelligence"):
            gr.Markdown("## Replenishment Risk Monitor\n**MCP Tool:** `low_stock_report()`")
            lowbtn=gr.Button("Generate Low-Stock Report",variant="primary")
            lowmsg=gr.Markdown(); lowdf=gr.Dataframe(interactive=False,wrap=True)

        with gr.Tab("Order Tracking"):
            gr.Markdown("## Verified Order Tracking\n**MCP Tool:** `get_order_status(order_id)`")
            with gr.Row():
                with gr.Column(scale=1):
                    oid=gr.Textbox(label="Order ID",placeholder="ORD-1002")
                    oidbtn=gr.Button("Track Order",variant="primary")
                    gr.Examples([["ORD-1001"],["ORD-1002"],["ORD-1003"]],oid)
                with gr.Column(scale=2): ordercard=gr.HTML("<div class='detail'>Enter an Order ID.</div>")
            orderraw=gr.Dataframe(label="Structured MCP Result",interactive=False)

        with gr.Tab("Restock Control"):
            gr.Markdown("## Controlled Replenishment\n**MCP Tool:** `create_restock_request(...)` — writes a pending record only; procurement remains under human approval.")
            with gr.Row():
                with gr.Column(scale=1):
                    rsku=gr.Textbox(label="SKU",value="KEY-310")
                    rqty=gr.Number(label="Requested Quantity",value=50,precision=0)
                    reason=gr.Textbox(label="Reason",value="Stock is at or below reorder level",lines=3)
                    rbtn=gr.Button("Create Pending Request",variant="primary")
                    gr.Markdown("> **Human-in-the-loop:** the AI can prepare a request, but it does not approve procurement.")
                with gr.Column(scale=2):
                    rstatus=gr.HTML("<div class='detail'>No request created yet.</div>")
                    rdf=gr.Dataframe(value=restock_table(),label="Restock Request Register",interactive=False,wrap=True)

        with gr.Tab("Support Assistant"):
            gr.Markdown("## Verified Customer Support Workspace\n**Tool → Resource → Prompt → Response**")
            with gr.Row():
                with gr.Column(scale=1):
                    soid=gr.Textbox(label="Order ID",value="ORD-1002")
                    issue=gr.Textbox(label="Customer Question",value="When will my order arrive, and can I return the laptop if I change my mind?",lines=5)
                    sbtn=gr.Button("Generate Verified Response",variant="primary")
                with gr.Column(scale=2): ans=gr.Markdown("Response will appear here.")
            with gr.Accordion("Show MCP evidence",open=False): evidence=gr.Markdown()

        with gr.Tab("MCP Explorer"):
            gr.Markdown("## MCP Capability Explorer\nDiscover Tools, Resources, Resource Templates, and Prompts dynamically.")
            ebtn=gr.Button("Discover MCP Capabilities",variant="primary")
            arch=gr.HTML(); edf=gr.Dataframe(interactive=False,wrap=True)

    demo.load(ui_low,outputs=[low_overview,low_summary])
    refresh.click(lambda:dashboard_html(),outputs=dashboard).then(ui_low,outputs=[low_overview,low_summary])
    qbtn.click(ui_search,q,[qdf,qmsg]); q.submit(ui_search,q,[qdf,qmsg])
    skubtn.click(ui_inventory,sku,[invcard,invraw]); sku.submit(ui_inventory,sku,[invcard,invraw])
    lowbtn.click(ui_low,outputs=[lowdf,lowmsg])
    oidbtn.click(ui_order,oid,[ordercard,orderraw]); oid.submit(ui_order,oid,[ordercard,orderraw])
    rbtn.click(ui_restock,[rsku,rqty,reason],[rstatus,rdf,dashboard])
    sbtn.click(ui_support,[soid,issue],[ans,evidence])
    ebtn.click(ui_explorer,outputs=[edf,arch])

print("Premium Gradio app built.")

/tmp/ipykernel_1284/2552625730.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme,css=CSS,title="RetailOps MCP Intelligence Console") as demo:


Premium Gradio app built.


# 8. Launch the App

For Google Colab, `share=True` is generally the easiest option. On local Jupyter you may use `share=False`.

In [ ]:
demo.queue().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7059be731556c33914.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Classroom Test Cases

### Search
Try `laptop`, `Accessory`, `CAM-610`.

### Inventory
Try `KEY-310`, `MON-220`, `CAM-610`.

### Orders
Try `ORD-1001`, `ORD-1002`, `ORD-1003`.

### Restock
Create a pending request for `KEY-310`, quantity `50`.

### MCP Explorer
Ask students to identify the Tools, Resources, Resource Template, and Prompts.

# Student Challenge

Add a **Management Brief** tab that:
1. calls `low_stock_report()`,
2. reads `retail://policy/restocking`,
3. renders `inventory_manager_brief(priority)`,
4. shows all evidence,
5. and never auto-approves a request.

**Bonus:** export the low-stock report as CSV.

# Key Takeaway

**MCP is the capability layer. Gradio is the experience layer.**

A strong AI application needs both an appealing interface and disciplined architecture for verified data access, safe tool execution, structured results, reusable context, error handling, and human oversight.